In [70]:
!pip install optuna

In [71]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [72]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [73]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [74]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [75]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-10-02 10:07:02,088] A new study created in memory with name: no-name-791715a2-7b20-42f8-b045-b976b189060b
[I 2026-10-02 10:07:02,713] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 147, 'max_depth': 20}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-02 10:07:03,212] Trial 1 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 134, 'max_depth': 4}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-02 10:07:03,985] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 175, 'max_depth': 4}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-02 10:07:04,523] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 103, 'max_depth': 17}. Best is trial 3 with value: 0.7746741154562384.
[I 2026-10-02 10:07:05,302] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 187, 'max_depth': 18}. Best is trial 3 with value: 0.774674

In [76]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7858472998137803
Best hyperparameters: {'n_estimators': 121, 'max_depth': 15}


In [77]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## Samplers in Optuna

In [78]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [79]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-10-02 10:07:36,468] A new study created in memory with name: no-name-1ecdf63c-5d9c-4274-8273-c05338af48b3
[I 2026-10-02 10:07:37,300] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 188, 'max_depth': 15}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-10-02 10:07:38,130] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 176, 'max_depth': 20}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-10-02 10:07:38,742] Trial 2 finished with value: 0.7821229050279329 and parameters: {'n_estimators': 124, 'max_depth': 18}. Best is trial 2 with value: 0.7821229050279329.
[I 2026-10-02 10:07:39,767] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 190, 'max_depth': 15}. Best is trial 2 with value: 0.7821229050279329.
[I 2026-10-02 10:07:40,436] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 68, 'max_depth': 3}. Best is trial 2 with value: 0.782122

In [80]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 124, 'max_depth': 18}


In [81]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [82]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [83]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-10-02 10:08:12,274] A new study created in memory with name: no-name-637979fe-cc1d-46c1-9231-76949d9b7a10
[I 2026-10-02 10:08:12,666] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-02 10:08:13,272] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-02 10:08:13,496] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-10-02 10:08:13,898] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-10-02 10:08:14,312] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [84]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [85]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [86]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [87]:
# 1. Optimization History
plot_optimization_history(study).show()

In [88]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [89]:
# 3. Slice Plot
plot_slice(study).show()

In [90]:
# 4. Contour Plot
plot_contour(study).show()

In [91]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [92]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [93]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [94]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-10-02 10:08:20,430] A new study created in memory with name: no-name-f32cdddf-f503-4a51-9277-7be9d3cff156
[I 2026-10-02 10:08:20,451] Trial 0 finished with value: 0.7132216014897579 and parameters: {'classifier': 'SVM', 'C': 0.14287233907892002, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7132216014897579.
[I 2026-10-02 10:08:22,875] Trial 1 finished with value: 0.7318435754189944 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 115, 'learning_rate': 0.03401688442413065, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.7318435754189944.
[I 2026-10-02 10:08:24,588] Trial 2 finished with value: 0.7169459962756052 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 108, 'learning_rate': 0.0305395340241417, 'max_depth': 14, 'min_samples_split': 9, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.7318435754189944.
[I 2026-10-02 10:08:26,296] Trial 3 finished with value: 0.743016

In [95]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11288928027838552, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [96]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.713222,2026-10-02 10:08:20.431704,2026-10-02 10:08:20.451440,0 days 00:00:00.019736,0.142872,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.731844,2026-10-02 10:08:20.452363,2026-10-02 10:08:22.875730,0 days 00:00:02.423367,NaN,NaN,GradientBoosting,NaN,NaN,0.034017,10.0,2.0,4.0,115.0,NaN,COMPLETE
2,2,0.716946,2026-10-02 10:08:22.876587,2026-10-02 10:08:24.588681,0 days 00:00:01.712094,NaN,NaN,GradientBoosting,NaN,NaN,0.030540,14.0,1.0,9.0,108.0,NaN,COMPLETE
3,3,0.743017,2026-10-02 10:08:24.589494,2026-10-02 10:08:26.296117,0 days 00:00:01.706623,NaN,NaN,GradientBoosting,NaN,NaN,0.295729,5.0,5.0,8.0,293.0,NaN,COMPLETE
4,4,0.785847,2026-10-02 10:08:26.296974,2026-10-02 10:08:26.363921,0 days 00:00:00.066947,17.180533,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.765363,2026-10-02 10:08:49.358342,2026-10-02 10:08:49.387315,0 days 00:00:00.028973,0.101175,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.785847,2026-10-02 10:08:49.388014,2026-10-02 10:08:49.408505,0 days 00:00:00.020491,0.207882,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
97,97,0.789572,2026-10-02 10:08:49.409270,2026-10-02 10:08:49.429420,0 days 00:00:00.020150,0.146160,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.724395,2026-10-02 10:08:49.430108,2026-10-02 10:08:49.451097,0 days 00:00:00.020989,0.268294,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [97]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,79
GradientBoosting,12
RandomForest,9


In [98]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.739292
RandomForest,0.765984
SVM,0.775358


In [99]:
# 1. Optimization History
plot_optimization_history(study).show()

In [100]:
# 3. Slice Plot
plot_slice(study).show()

In [101]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [102]:
!pip install optuna-integration[xgboost]


In [103]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-10-02 10:08:54,712] A new study created in memory with name: no-name-45043a9c-c88e-4172-b863-e065c05a1411


[0]	train-mlogloss:1.00093	eval-mlogloss:0.99805
[1]	train-mlogloss:0.90937	eval-mlogloss:0.90414
[2]	train-mlogloss:0.86877	eval-mlogloss:0.87191
[3]	train-mlogloss:0.83440	eval-mlogloss:0.83640
[4]	train-mlogloss:0.76577	eval-mlogloss:0.76474
[5]	train-mlogloss:0.69586	eval-mlogloss:0.69170
[6]	train-mlogloss:0.67350	eval-mlogloss:0.66791
[7]	train-mlogloss:0.64665	eval-mlogloss:0.63703
[8]	train-mlogloss:0.59645	eval-mlogloss:0.58294
[9]	train-mlogloss:0.54703	eval-mlogloss:0.52692
[10]	train-mlogloss:0.51308	eval-mlogloss:0.48743
[11]	train-mlogloss:0.48075	eval-mlogloss:0.45249
[12]	train-mlogloss:0.45038	eval-mlogloss:0.42224
[13]	train-mlogloss:0.43107	eval-mlogloss:0.40298
[14]	train-mlogloss:0.40579	eval-mlogloss:0.37579
[15]	train-mlogloss:0.38212	eval-mlogloss:0.34921
[16]	train-mlogloss:0.36096	eval-mlogloss:0.32528
[17]	train-mlogloss:0.33420	eval-mlogloss:0.29694
[18]	train-mlogloss:0.31532	eval-mlogloss:0.27680
[19]	train-mlogloss:0.30063	eval-mlogloss:0.26013
[20]	train

[I 2026-10-02 10:08:55,312] Trial 0 finished with value: 1.0 and parameters: {'lambda': 1.2228682677245145e-06, 'alpha': 0.001356651354882829, 'eta': 0.10290696426865867, 'gamma': 6.497256432362295e-06, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.7562278153728188, 'colsample_bytree': 0.4121805001705988}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.04946	eval-mlogloss:1.04728
[1]	train-mlogloss:1.00396	eval-mlogloss:0.99939
[2]	train-mlogloss:0.96100	eval-mlogloss:0.95300
[3]	train-mlogloss:0.92126	eval-mlogloss:0.91252
[4]	train-mlogloss:0.88213	eval-mlogloss:0.87159
[5]	train-mlogloss:0.84642	eval-mlogloss:0.83480
[6]	train-mlogloss:0.81301	eval-mlogloss:0.79917
[7]	train-mlogloss:0.78141	eval-mlogloss:0.76665
[8]	train-mlogloss:0.75092	eval-mlogloss:0.73457
[9]	train-mlogloss:0.72210	eval-mlogloss:0.70381
[10]	train-mlogloss:0.69479	eval-mlogloss:0.67539
[11]	train-mlogloss:0.66944	eval-mlogloss:0.64869
[12]	train-mlogloss:0.64450	eval-mlogloss:0.62269
[13]	train-mlogloss:0.62106	eval-mlogloss:0.59785
[14]	train-mlogloss:0.59834	eval-mlogloss:0.57573
[15]	train-mlogloss:0.57650	eval-mlogloss:0.55304
[16]	train-mlogloss:0.55611	eval-mlogloss:0.53126
[17]	train-mlogloss:0.53706	eval-mlogloss:0.51104
[18]	train-mlogloss:0.51827	eval-mlogloss:0.49124
[19]	train-mlogloss:0.50093	eval-mlogloss:0.47351
[20]	train

[I 2026-10-02 10:08:55,992] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.007080610727302091, 'alpha': 0.0005400665061186119, 'eta': 0.03741815191721627, 'gamma': 9.55155223337928e-05, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.6765911656580488, 'colsample_bytree': 0.8292703498453438}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.07994	eval-mlogloss:1.08278
[1]	train-mlogloss:1.06178	eval-mlogloss:1.06442
[2]	train-mlogloss:1.05148	eval-mlogloss:1.05790
[3]	train-mlogloss:1.04137	eval-mlogloss:1.05026
[4]	train-mlogloss:1.02411	eval-mlogloss:1.03284
[5]	train-mlogloss:1.00541	eval-mlogloss:1.01377
[6]	train-mlogloss:0.99608	eval-mlogloss:1.00521
[7]	train-mlogloss:0.98517	eval-mlogloss:0.99522
[8]	train-mlogloss:0.96900	eval-mlogloss:0.97913
[9]	train-mlogloss:0.95139	eval-mlogloss:0.96091
[10]	train-mlogloss:0.93720	eval-mlogloss:0.94635
[11]	train-mlogloss:0.92274	eval-mlogloss:0.93128
[12]	train-mlogloss:0.90861	eval-mlogloss:0.91713
[13]	train-mlogloss:0.89734	eval-mlogloss:0.90734
[14]	train-mlogloss:0.88345	eval-mlogloss:0.89364
[15]	train-mlogloss:0.86984	eval-mlogloss:0.87986
[16]	train-mlogloss:0.85697	eval-mlogloss:0.86648
[17]	train-mlogloss:0.84073	eval-mlogloss:0.84954
[18]	train-mlogloss:0.82761	eval-mlogloss:0.83619
[19]	train-mlogloss:0.81445	eval-mlogloss:0.82351
[20]	train

[I 2026-10-02 10:09:23,449] Trial 48 pruned. Trial was pruned at iteration 32.


[0]	train-mlogloss:1.06422	eval-mlogloss:1.06664
[1]	train-mlogloss:1.03086	eval-mlogloss:1.03241


[I 2026-10-02 10:09:24,184] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 1.2228682677245145e-06, 'alpha': 0.001356651354882829, 'eta': 0.10290696426865867, 'gamma': 6.497256432362295e-06, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.7562278153728188, 'colsample_bytree': 0.4121805001705988}
Best accuracy: 1.0


In [104]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()